# Challenge 2 – Classification into screws and nuts
**Group:** (names) &nbsp; **N =** (2 or 3)

Input: label images from Challenge 1 (`outputs/c1/<method>/`) and ground truth (`annotations/parts.csv` + `<stem>_labels.png`).
Steps: (1) per-part descriptors → (2) descriptor study (distributions, discriminative power, invariance, redundancy) →
(3) classifiers, evaluated with **cross-validation grouped by source image** → (4) behaviour under imperfect segmentation → (5) final model applied to every image in `inputs/`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage import measure
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from sklearn.feature_selection import mutual_info_classif

SEED = 0; np.random.seed(SEED)
IN, ANN, C1, OUT = Path("inputs"), Path("annotations"), Path("outputs/c1"), Path("outputs/c2")
OUT.mkdir(parents=True, exist_ok=True)
EXT = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

# ---- parameters (justify in the report) ----
MAX_SIDE      = 1200     # must equal MAX_SIDE in the Challenge 1 notebook
USE_AUG       = False    # include augmented images (<stem>__augN) in the analysis
MIN_PART_PX   = 30       # ignore components smaller than this
BEST_SEG      = "otsu"   # segmentation used for the final run (decision of Challenge 1)
FEATSET_MAIN  = "invariant"   # "invariant" (rotation+scale) or "rot_inv" (rotation only, allows size cues)

## 1. Ground truth and descriptors

In [ ]:
assert (ANN / "parts.csv").exists(), "annotations/parts.csv missing (see make_parts_template.py)"
parts = pd.read_csv(ANN / "parts.csv")
parts["stem"] = parts["image"].astype(str).map(lambda s: Path(s).stem)
parts["type"] = parts["type"].astype(str).str.strip().str.lower()
type_map = {(r.stem, int(r.part_id)): r.type for r in parts.itertuples() if r.type in ("screw", "nut")}
src = lambda s: s.split("__aug")[0]
def gt_type(stem, pid):                      # augmented images keep the part ids of their source image
    return type_map.get((stem, pid), type_map.get((src(stem), pid)))

stems = sorted(p.stem for p in IN.iterdir() if p.suffix.lower() in EXT
               and (ANN / f"{p.stem}_labels.png").exists() and (USE_AUG or "__aug" not in p.stem))
def load_gt(stem):
    g = cv2.imread(str(ANN / f"{stem}_labels.png"), cv2.IMREAD_UNCHANGED); g = g[..., 0] if g.ndim == 3 else g
    s = min(1.0, MAX_SIDE / max(g.shape)); W, H = int(g.shape[1]*s), int(g.shape[0]*s)
    return cv2.resize(g, (W, H), interpolation=cv2.INTER_NEAREST).astype(np.int32)
GT = {s: load_gt(s) for s in stems}
print(len(stems), "annotated images;", parts.type.value_counts().to_dict())

In [ ]:
HOLE_MIN = 3
def part_features(mask):
    """Descriptors of ONE part (boolean mask). Holes are kept for topology; shape descriptors use the filled silhouette."""
    ys, xs = np.nonzero(mask)
    m = np.pad(mask[ys.min():ys.max()+1, xs.min():xs.max()+1], 3)
    filled = ndi.binary_fill_holes(m)
    area, farea = float(m.sum()), float(filled.sum())
    cnts, _ = cv2.findContours(filled.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    c = max(cnts, key=cv2.contourArea)
    per = max(cv2.arcLength(c, True), 1.); hull = cv2.convexHull(c); hper = cv2.arcLength(hull, True); harea = max(cv2.contourArea(hull), 1.)
    (_, _), (w, h), _ = cv2.minAreaRect(c); L, Wd = max(w, h, 1.), max(min(w, h), 1.)
    holes = [r for r in measure.regionprops(measure.label(filled & ~m, connectivity=1)) if r.area >= HOLE_MIN]
    cy, cx = ndi.center_of_mass(filled); req = np.sqrt(farea / np.pi)
    if holes:
        big = max(holes, key=lambda r: r.area)
        hfrac, hoff = big.area / farea, np.hypot(big.centroid[0]-cy, big.centroid[1]-cx) / req
    else: hfrac = hoff = 0.
    rp = measure.regionprops(filled.astype(np.uint8))[0]
    hu = cv2.HuMoments(cv2.moments(filled.astype(np.uint8), binaryImage=True)).ravel()
    hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-30)
    dt = ndi.distance_transform_edt(filled)
    d = dict(area=area, filled_area=farea, perimeter=per, major_len=L, minor_len=Wd,          # scale-dependent
             aspect=L/Wd, rect_fill=farea/(L*Wd), solidity=farea/harea, convexity=hper/per,
             circularity=4*np.pi*farea/per**2, eccentricity=rp.eccentricity, extent=rp.extent,
             fill_ratio=area/farea, euler=float(measure.euler_number(m, connectivity=2)), n_holes=float(len(holes)),
             hole_frac=hfrac, hole_offset=hoff, inscribed_ratio=2*dt.max()/L,
             n_vert=float(len(cv2.approxPolyDP(hull, 0.03*hper, True))))
    d.update({f"hu{i+1}": hu[i] for i in range(7)})
    return d

SCALE_DEP = ["area", "filled_area", "perimeter", "major_len", "minor_len"]
FEATS = list(part_features(np.pad(np.ones((20, 40), bool), 5)).keys())
print(len(FEATS), "descriptors:", FEATS)

In [ ]:
MIN_IOU = 0.5
def build_table(stem, lab, gt):
    """One row per component of `lab`; its true type comes from the GT part it overlaps (IoU >= 0.5), else 'fp'."""
    rows = []; ga = {int(i): int((gt == i).sum()) for i in np.unique(gt[gt > 0])}
    for i in np.unique(lab[lab > 0]):
        m = lab == i; a = int(m.sum())
        if a < MIN_PART_PX: continue
        ids, cnt = np.unique(gt[m & (gt > 0)], return_counts=True)
        if len(ids): j = int(ids[np.argmax(cnt)]); iou = cnt.max() / (a + ga[j] - cnt.max())
        else: j, iou = 0, 0.
        t = gt_type(stem, j) if iou >= MIN_IOU else "fp"
        if t is None: continue
        rows.append(dict(image=stem, source=src(stem), comp=int(i), gt_id=j, iou=iou, type=t, **part_features(m)))
    return rows

def perturb(gt, fn):
    out = np.zeros_like(gt)
    for i in np.unique(gt[gt > 0]):
        pm = fn(gt == i)
        if pm.sum() > 0: out[pm & (out == 0)] = i
    return out
ell = lambda r: cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2*r+1, 2*r+1))
PERTURB = {"lost_holes": lambda m: ndi.binary_fill_holes(m),
           "eroded_2px": lambda m: cv2.erode(m.astype(np.uint8), ell(2)) > 0,
           "dilated_2px": lambda m: cv2.dilate(m.astype(np.uint8), ell(2)) > 0}

TABLES = {"gt": pd.DataFrame([r for s in stems for r in build_table(s, GT[s], GT[s])])}
for name, fn in PERTURB.items():
    TABLES["perturb:" + name] = pd.DataFrame([r for s in stems for r in build_table(s, perturb(GT[s], fn), GT[s])])
SEG_METHODS = sorted(d.name for d in C1.iterdir() if d.is_dir()) if C1.exists() else []
for meth in SEG_METHODS:
    rows = []
    for s in stems:
        f = C1 / meth / f"{s}_labels.png"
        if f.exists(): rows += build_table(s, cv2.imread(str(f), cv2.IMREAD_UNCHANGED).astype(np.int32), GT[s])
    TABLES["seg:" + meth] = pd.DataFrame(rows)
df = TABLES["gt"]; df.to_csv(OUT / "features_gt.csv", index=False)
print({k: len(v) for k, v in TABLES.items()}); print(df.type.value_counts().to_dict(), "| sources:", df.source.nunique())

## 2. Descriptor study
### 2.1 Distribution per class

In [ ]:
y = (df.type == "screw").astype(int).values
n = len(FEATS); cols = 6; rws = int(np.ceil(n / cols))
fig, ax = plt.subplots(rws, cols, figsize=(3*cols, 2.2*rws))
for a, f in zip(ax.ravel(), FEATS):
    v = df[f].replace([np.inf, -np.inf], np.nan).dropna()
    bins = np.histogram_bin_edges(v, 20)
    for cls, col in (("nut", "tab:blue"), ("screw", "tab:red")):
        a.hist(df.loc[df.type == cls, f].replace([np.inf, -np.inf], np.nan).dropna(), bins=bins, alpha=.6, color=col, label=cls)
    a.set_title(f, fontsize=9); a.tick_params(labelsize=6)
ax.ravel()[0].legend(fontsize=7)
for a in ax.ravel()[n:]: a.axis("off")
plt.tight_layout(); plt.savefig(OUT / "feature_distributions.png", dpi=130); plt.show()

### 2.2 Discriminative power (single-feature AUC, Fisher ratio, mutual information)

In [ ]:
X = df[FEATS].replace([np.inf, -np.inf], np.nan).fillna(0)
stat = pd.DataFrame(index=FEATS)
stat["auc"] = [max(roc_auc_score(y, X[f]), 1 - roc_auc_score(y, X[f])) for f in FEATS]     # direction-free
stat["fisher"] = [(X[f][y==1].mean() - X[f][y==0].mean())**2 / (X[f][y==1].var() + X[f][y==0].var() + 1e-12) for f in FEATS]
stat["mutual_info"] = mutual_info_classif(X, y, random_state=SEED)
stat = stat.sort_values("auc", ascending=False); display(stat.round(3))

### 2.3 Invariance to rotation and scale
Each GT mask is rotated (scale 1) and rescaled (angle 0) with known factors, descriptors are recomputed, and
**instability = mean within-part std of the descriptor / its std across parts** (0 = invariant, ≳0.2 = not usable).
Note: rescaling very small masks closes their bores – this is itself a finding.

In [ ]:
def transform_mask(m, angle, scale):
    ys, xs = np.nonzero(m); crop = m[ys.min():ys.max()+1, xs.min():xs.max()+1].astype(np.float32)
    h, w = crop.shape; d = int(np.hypot(h, w) * max(scale, 1)) + 6
    canvas = np.zeros((d, d), np.float32); y0, x0 = (d - h)//2, (d - w)//2; canvas[y0:y0+h, x0:x0+w] = crop
    M = cv2.getRotationMatrix2D((d/2, d/2), angle, scale)
    return cv2.warpAffine(canvas, M, (d, d), flags=cv2.INTER_LINEAR) > 0.5

ANGLES, SCALES = [0, 17, 45, 60, 90, 133, 200, 290], [0.6, 0.8, 1.0, 1.25, 1.5]
sel_rows = df.sample(min(40, len(df)), random_state=SEED)
def inst(kind):
    per_part = []
    for r in sel_rows.itertuples():
        m = GT[r.image] == r.comp
        vals = [part_features(transform_mask(m, a, 1.0) if kind == "rot" else transform_mask(m, 0, s)) for a in (ANGLES if kind == "rot" else [0]) for s in ([1.0] if kind == "rot" else SCALES)]
        per_part.append(pd.DataFrame(vals)[FEATS].replace([np.inf, -np.inf], np.nan).std())
    return pd.concat(per_part, axis=1).mean(axis=1) / (X[FEATS].std() + 1e-12)
invar = pd.DataFrame({"instab_rot": inst("rot"), "instab_scale": inst("scale")}).join(stat[["auc"]])
invar.to_csv(OUT / "invariance.csv"); display(invar.sort_values("instab_rot").round(3))

### 2.4 Redundancy and selected subsets

In [ ]:
corr = X[FEATS].corr(method="spearman")
fig, ax = plt.subplots(figsize=(9, 8)); im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(FEATS))); ax.set_xticklabels(FEATS, rotation=90, fontsize=6); ax.set_yticks(range(len(FEATS))); ax.set_yticklabels(FEATS, fontsize=6)
plt.colorbar(im); plt.tight_layout(); plt.savefig(OUT / "feature_correlation.png", dpi=130); plt.show()

def greedy_select(cands, thr=0.9, kmax=6, auc_min=0.8):
    sel = []
    for f in sorted(cands, key=lambda f: -stat.loc[f, "auc"]):
        if stat.loc[f, "auc"] < auc_min: break          # do not pad the subset with weak descriptors
        if all(abs(corr.loc[f, s]) < thr for s in sel): sel.append(f)
        if len(sel) >= kmax: break
    return sel
TH = 0.15   # instability tolerated
rot_ok = [f for f in FEATS if invar.loc[f, "instab_rot"] < TH]
both_ok = [f for f in rot_ok if invar.loc[f, "instab_scale"] < TH]
FEATSETS = {"invariant": greedy_select(both_ok), "rot_inv": greedy_select(rot_ok)}
for k, v in FEATSETS.items(): print(f"{k:10s}", v)

## 3. Classifiers (≥ N of different nature)
Nested evaluation: outer **GroupKFold by source image** (no part of an image is in both train and test); hyper-parameters tuned by an inner grouped CV (macro-F1).
Justify each model and grid in the report: tree = interpretable rules, k-NN = non-parametric, SVM = margin/kernel, logistic regression & LDA = linear, RF = ensemble.

In [ ]:
CLFS = {
 "tree":   (DecisionTreeClassifier(class_weight="balanced", random_state=SEED), {"max_depth": [1, 2, 3, 4]}),
 "knn":    (make_pipeline(StandardScaler(), KNeighborsClassifier()), {"kneighborsclassifier__n_neighbors": [1, 3, 5, 7]}),
 "svm":    (make_pipeline(StandardScaler(), SVC(class_weight="balanced", random_state=SEED)), {"svc__C": [0.1, 1, 10], "svc__gamma": ["scale", 0.1]}),
 "logreg": (make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=2000)), {"logisticregression__C": [0.01, 0.1, 1, 10]}),
 "lda":    (make_pipeline(StandardScaler(), LinearDiscriminantAnalysis()), {}),
 "rf":     (RandomForestClassifier(200, class_weight="balanced", random_state=SEED), {"max_depth": [2, 4, None]}),
}
def run_cv(fs_name, feats):
    srcs = df.source.nunique(); gkf = GroupKFold(n_splits=min(5, srcs)); out = []
    for fold, (tr, te) in enumerate(gkf.split(df, df.type, df.source)):
        trd = df.iloc[tr]; te_src = set(df.iloc[te].source)
        Xtr = trd[feats].replace([np.inf, -np.inf], 0).values; ytr = trd.type.values
        for cname, (est, grid) in CLFS.items():
            if grid:
                gs = GridSearchCV(est, grid, cv=GroupKFold(min(3, trd.source.nunique())), scoring="f1_macro")
                model = gs.fit(Xtr, ytr, groups=trd.source.values).best_estimator_
            else: model = est.fit(Xtr, ytr)
            for cond, tab in TABLES.items():
                sub = tab[tab.source.isin(te_src)]
                if len(sub) == 0: continue
                p = model.predict(sub[feats].replace([np.inf, -np.inf], 0).values)
                out.append(pd.DataFrame(dict(clf=cname, featset=fs_name, cond=cond, fold=fold, image=sub.image.values, comp=sub.comp.values, type=sub.type.values, pred=p)))
    return pd.concat(out)
PRED = pd.concat([run_cv(k, v) for k, v in FEATSETS.items()]); PRED.to_csv(OUT / "cv_predictions.csv", index=False)

def summarize(P):
    gtc = TABLES["gt"].groupby(["image", "type"]).size().unstack(fill_value=0).reindex(columns=["nut", "screw"], fill_value=0)
    rows = []
    for (c, fs, cond), g in P.groupby(["clf", "featset", "cond"]):
        m = g[g.type != "fp"]; fp = g[g.type == "fp"]
        pc = g.groupby(["image", "pred"]).size().unstack(fill_value=0).reindex(columns=["nut", "screw"], fill_value=0)
        diff = (pc - gtc.reindex(pc.index).fillna(0)).abs().mean()
        rows.append(dict(clf=c, featset=fs, cond=cond, n=len(m), accuracy=(m.type == m.pred).mean(),
                         f1_macro=f1_score(m.type, m.pred, average="macro"), fp_comps=len(fp), fp_as_screw=int((fp.pred == "screw").sum()),
                         fp_as_nut=int((fp.pred == "nut").sum()), count_mae_nut=diff["nut"], count_mae_screw=diff["screw"]))
    return pd.DataFrame(rows)
RES = summarize(PRED); RES.to_csv(OUT / "results.csv", index=False)
display(RES[RES.cond == "gt"].pivot(index="clf", columns="featset", values=["accuracy", "f1_macro"]).round(3))

### 3.1 Errors and interpretable rules on clean (ground-truth) masks

In [ ]:
main = FEATSETS[FEATSET_MAIN]
G = PRED[(PRED.cond == "gt") & (PRED.featset == FEATSET_MAIN)]
errs = G[G.type != G.pred].merge(df[["image", "comp"] + main], on=["image", "comp"])
print("misclassified parts per classifier:"); display(G.assign(err=G.type != G.pred).groupby("clf").err.sum().to_frame("errors"))
display(errs.sort_values(["image", "comp"]).round(3).head(30))
tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=SEED).fit(df[main].replace([np.inf, -np.inf], 0), df.type)
print(export_text(tree, feature_names=main))

## 4. Behaviour under imperfect segmentation
`perturb:*` = controlled damage of the GT masks (lost bores, ±2 px boundary error); `seg:*` = real Challenge 1 outputs.
Columns: accuracy on matched components, **false-positive components** and what they are classified as, and the error in the number of nuts / screws per image.

In [ ]:
R = RES[RES.featset == FEATSET_MAIN].copy()
display(R.pivot(index="clf", columns="cond", values="accuracy").round(3))
display(R[R.cond.str.startswith("seg:")][["clf", "cond", "fp_comps", "fp_as_screw", "fp_as_nut", "count_mae_nut", "count_mae_screw"]].round(3).sort_values(["cond", "clf"]))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.5)); piv = R.pivot(index="cond", columns="clf", values="accuracy").loc[[c for c in TABLES]]
piv.plot.bar(ax=ax); ax.set_ylabel("accuracy (matched parts)"); ax.set_ylim(max(0, piv.min().min() - .05), 1.01); plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.savefig(OUT / "robustness.png", dpi=130); plt.show()

## 5. Final model → every image in `inputs/`

In [ ]:
best = RES[(RES.cond == "gt") & (RES.featset == FEATSET_MAIN)].sort_values(["f1_macro", "accuracy"], ascending=False).iloc[0].clf
print("final classifier:", best, "| features:", main)
est, grid = CLFS[best]
Xall, yall = df[main].replace([np.inf, -np.inf], 0).values, df.type.values
final = (GridSearchCV(est, grid, cv=GroupKFold(min(5, df.source.nunique())), scoring="f1_macro").fit(Xall, yall, groups=df.source.values).best_estimator_
         if grid else est.fit(Xall, yall))
seg = BEST_SEG if (C1 / BEST_SEG).exists() else SEG_METHODS[0]
allimgs = sorted(p for p in IN.iterdir() if p.suffix.lower() in EXT)
res_rows, shown = [], []
for p in allimgs:
    f = C1 / seg / f"{p.stem}_labels.png"
    if not f.exists(): continue
    lab = cv2.imread(str(f), cv2.IMREAD_UNCHANGED).astype(np.int32); comps = []
    for i in np.unique(lab[lab > 0]):
        m = lab == i
        if m.sum() < MIN_PART_PX: continue
        ft = part_features(m); comps.append((int(i), ft))
    if comps:
        pr = final.predict(pd.DataFrame([c[1] for c in comps])[main].replace([np.inf, -np.inf], 0).values)
    else: pr = []
    for (i, _), c in zip(comps, pr): res_rows.append(dict(image=p.name, comp=i, cls=c))
    shown.append((p, lab, dict(zip([c[0] for c in comps], pr))))
out = pd.DataFrame(res_rows); out.to_csv(OUT / "classification.csv", index=False)
cnt = out.groupby(["image", "cls"]).size().unstack(fill_value=0).reindex(columns=["nut", "screw"], fill_value=0); cnt.to_csv(OUT / "counts_nuts_screws.csv"); display(cnt)
sel = shown[:6]; fig, ax = plt.subplots(2, 3, figsize=(15, 8))
for a, (p, lab, cm) in zip(ax.ravel(), sel):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB); img = cv2.resize(img, (lab.shape[1], lab.shape[0]))
    ov = img.copy()
    for i, c in cm.items(): ov[lab == i] = (0.4*ov[lab == i] + 0.6*np.array([255, 0, 0] if c == "screw" else [0, 80, 255])).astype(np.uint8)
    a.imshow(ov); a.set_title(f"{p.name}: {list(cm.values()).count('nut')} nuts, {list(cm.values()).count('screw')} screws", fontsize=9); a.axis("off")
plt.tight_layout(); plt.savefig(OUT / "classification_overlay.png", dpi=120); plt.show()

## 6. To discuss in the report (your own words)
- Which descriptors separate the classes, and *why* (bore / topology, aspect, solidity, inscribed ratio, n_vert)? Distribution plots + AUC table.
- Which descriptors are not rotation/scale invariant? Cost/benefit of allowing size cues (`rot_inv`) on a fixed-camera workstation.
- Redundancy: which descriptors collapse together, and what subset did you keep (thresholds `TH`, correlation 0.9)?
- Per classifier: justification, hyper-parameters, CV results, errors. Is the difference between classifiers significant with so few parts?
- Imperfect segmentation: lost bores (`perturb:lost_holes`; small nuts in `screws_028`), chips as false positives (`screws_061`), merged parts. Which classifier degrades least and why?
- Limits: tiny dataset, correlated parts of the same image, annotation noise (`notes` column).